In [1]:
%reset -f

In [2]:
# Standard imports
import sys
sys.path.insert(0, '/home/ubuntu/prem')

import os
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
import geopandas as gpd
from shapely import wkt
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Modular imports
from utils import log_memory, log_df_memory, save_detection_results
from filters.preprocessing import (
    filter_by_lifetime,
    attach_zones_to_objects,
    apply_footpath_zone_filter,
    compute_polygon_orientation,
    filter_parallel_vehicles,
    filter_static_objects
)
from regions.oulu.zones import (
    get_crosswalk_zone, 
    get_footpath_zones, 
    get_near_miss_zones, 
    get_exclusion_zone
)
from ssm.utils import load_config, find_all_nearby_pairs, get_mdrac_pairs
from ssm.m_drac import ModifiedDRAC

[SSM] Numba enabled with 7 threads


In [4]:
# Configuration
START_DATE = "2025-08-22"
END_DATE = "2025-09-11"
DATA_DIR = "/home/ubuntu/data/uploads/oulu_data/objects/clean/objects/clean"
OUTPUT_DIR = "/home/ubuntu/prem/results"

config = load_config("/home/ubuntu/prem/config.yaml")

print("="*70)
print("OULU PEDESTRIAN CROSSING ANALYSIS")
print("="*70)
print(f"Date: {START_DATE} to {END_DATE}")
print("="*70)

OULU PEDESTRIAN CROSSING ANALYSIS
Date: 2025-08-22 to 2025-09-11


## 1. Data Loading

In [5]:
def load_oulu_data(data_dir, start_date, end_date):
    """
    Load Oulu data from hourly parquet folders.
    
    Data structure: YYYY-MM-DD-HH/YYYY-MM-DD-HH-MM.parquet
    """
    dfs = []
    
    # List all folders
    folders = sorted([f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))])
    
    for folder in tqdm(folders, desc="Loading data"):
        # Extract date from folder name (YYYY-MM-DD-HH)
        try:
            folder_date = folder[:10]  # YYYY-MM-DD
            
            # Check if within date range
            if folder_date < start_date or folder_date > end_date:
                continue
            
            folder_path = os.path.join(data_dir, folder)
            
            # Load all parquet files in this hour folder
            df_hour = pd.read_parquet(folder_path)
            dfs.append(df_hour)
            
        except Exception as e:
            print(f"Error loading {folder}: {e}")
            continue
    
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        print(f"\n✓ Loaded {len(df):,} records from {len(dfs)} hour folders")
        return df
    else:
        print("No data found for given date range.")
        return pd.DataFrame()

In [6]:
print("\nLoading Oulu data...")
log_memory("Before loading")

df = load_oulu_data(DATA_DIR, START_DATE, END_DATE)

log_df_memory(df, "Loaded data")
df.reset_index(drop=True, inplace=True)
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")


Loading Oulu data...
[MEMORY] Before loading: 194.4 MB


Loading data: 100%|██████████| 617/617 [00:06<00:00, 100.42it/s]



✓ Loaded 43,174,533 records from 481 hour folders
[DF MEMORY] Loaded data: 2635.2 MB (43,174,533 rows)

Date range: 2025-08-22 23:45:00.020003796 to 2025-09-11 23:59:59.922009945


## 2. Lifetime Filtering

In [7]:
print("\n" + "="*70)
print("Lifetime Filtering")
print("="*70)

df = filter_by_lifetime(df, config['preprocessing']['lifetime_filter']['min_lifespan'])
log_df_memory(df, "After lifetime filter")


Lifetime Filtering
[lifespan filter] Removed 280,488 short-lived IDs
  Before: 465,528 IDs (43,174,533 rows)
  After: 185,040 IDs (28,968,836 rows)
[DF MEMORY] After lifetime filter: 1989.1 MB (28,968,836 rows)


np.float64(1989.1321105957031)

## 3. Footpath Zone Filtering

In [8]:
print("\n" + "="*70)
print("Footpath Zone Filtering")
print("="*70)

footpath_zones = get_footpath_zones()
zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

print(f"Attaching footpath zones to {len(df):,} rows...")
log_memory("Before footpath zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After footpath zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

df = apply_footpath_zone_filter(df)
df = df.drop(columns=['zone'], errors='ignore')
df.reset_index(drop=True, inplace=True)
gc.collect()
log_memory("After footpath filter")


Footpath Zone Filtering
Attaching footpath zones to 28,968,836 rows...
[MEMORY] Before footpath zones: 5037.2 MB


Zone assignment batches: 100%|██████████| 290/290 [00:39<00:00,  7.39it/s]


[MEMORY] After footpath zones: 7004.1 MB
✓ Zones attached! Total rows: 28,968,836
[footpath zone] Removed 25501 objects
[MEMORY] After footpath filter: 6259.0 MB


6258.9765625

## 4. Crosswalk Zone Filtering

In [9]:
print("\n" + "="*70)
print("Crosswalk Zone Filtering (Remove Parallel Cars)")
print("="*70)

crosswalk_zone = get_crosswalk_zone()
zones_df = pd.DataFrame([crosswalk_zone])
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

print(f"Crosswalk orientation: {gdf_zones['orientation_deg'].iloc[0]:.2f}°")
print(f"Attaching crosswalk zone to {len(df):,} rows...")
log_memory("Before crosswalk zones")

df = attach_zones_to_objects(df, gdf_zones, how="left", batch_size=100000)

log_memory("After crosswalk zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

# Filter parallel vehicles (cars moving along crosswalk, not crossing)
removed_ids_global = []
df_in_zones = df[df['zone'].notnull()].copy()

if len(df_in_zones) > 0:
    orientation = gdf_zones['orientation_deg'].iloc[0]
    parallel_ids, _ = filter_parallel_vehicles(df_in_zones, orientation, threshold=4.0)
    removed_ids_global.extend(parallel_ids)
    df = df[~df['id'].isin(removed_ids_global)]
    print(f"[crosswalk] Removed {len(removed_ids_global):,} parallel vehicles")

df = df.drop(columns=['zone'], errors='ignore')
df.reset_index(drop=True, inplace=True)
gc.collect()
log_memory("After crosswalk filter")


Crosswalk Zone Filtering (Remove Parallel Cars)
Crosswalk orientation: -88.57°
Attaching crosswalk zone to 26,168,088 rows...
[MEMORY] Before crosswalk zones: 6259.0 MB


Zone assignment batches: 100%|██████████| 262/262 [00:34<00:00,  7.69it/s]


[MEMORY] After crosswalk zones: 6561.8 MB
✓ Zones attached! Total rows: 26,168,088
[crosswalk] Removed 25 parallel vehicles
[MEMORY] After crosswalk filter: 6566.9 MB


6566.94140625

## 5. Static Object Removal

In [10]:
print("\n" + "="*70)
print("Static Object Removal")
print("="*70)

df = filter_static_objects(df, 
    static_threshold=config['preprocessing']['static_filter']['min_speed'],
    static_ratio_min=0.8)

log_df_memory(df, "After static filter")


Static Object Removal
[static filter] Found 4,582 static objects
[static filter] Removed 4,582 static objects
  Before: 159,514 IDs (26,164,522 rows)
  After: 154,932 IDs (22,477,205 rows)
[DF MEMORY] After static filter: 1543.4 MB (22,477,205 rows)


np.float64(1543.3871841430664)

## 6. Exclusion Zone Filtering

In [11]:
print("\n" + "="*70)
print("Exclusion Zone Filtering")
print("="*70)

exclusion_zone = get_exclusion_zone()
exclusion_poly = wkt.loads(exclusion_zone["vertices"])

# VECTORIZED APPROACH - Much faster than df.apply()
from shapely.geometry import Point

# Create GeoDataFrame with point geometries
gdf_temp = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['pos_x'], df['pos_y'])
)

# Vectorized contains check (much faster than row-by-row apply)
df['in_exclusion_zone'] = gdf_temp.geometry.within(exclusion_poly)

removed = df['in_exclusion_zone'].sum()

df = df[~df['in_exclusion_zone']].copy()
df.drop(columns=['in_exclusion_zone'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"[exclusion zone] Removed {removed:,} observations")
log_df_memory(df, "After exclusion filter")


Exclusion Zone Filtering
[exclusion zone] Removed 4,503 observations
[DF MEMORY] After exclusion filter: 1371.6 MB (22,472,702 rows)


np.float64(1371.625)

## 7. Near-Miss Zone Assignment

In [12]:
print("\n" + "="*70)
print("Near-Miss Zone Assignment")
print("="*70)

near_miss_zones = get_near_miss_zones()
zones_df = pd.DataFrame(near_miss_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

print(f"Attaching near-miss zones to {len(df):,} rows...")
log_memory("Before near-miss zones")

df = attach_zones_to_objects(df, gdf_zones, how="inner", batch_size=100000)

log_memory("After near-miss zones")
print(f"✓ Zones attached! Total rows: {len(df):,}")

print("\nZone distribution:")
print(df['zone'].value_counts())

df_analysis = df.copy()
log_df_memory(df_analysis, "Analysis-ready data")


Near-Miss Zone Assignment
Attaching near-miss zones to 22,472,702 rows...
[MEMORY] Before near-miss zones: 11046.0 MB


Zone assignment batches: 100%|██████████| 225/225 [01:06<00:00,  3.36it/s]


[MEMORY] After near-miss zones: 10375.4 MB
✓ Zones attached! Total rows: 4,704,167

Zone distribution:
zone
1002    2611333
1001    2092834
Name: count, dtype: int64
[DF MEMORY] Analysis-ready data: 291.6 MB (4,704,167 rows)


np.float64(291.60615253448486)

## 8. M-DRAC Detection

In [13]:
print("\n" + "="*70)
print("M-DRAC Detection")
print("="*70)

# Generate base pairs
print("\nGenerating nearby pairs...")
log_memory("Before pair generation")

base_pairs = find_all_nearby_pairs(df_analysis, config)

print(f"✓ Generated {len(base_pairs):,} base pairs")
log_memory("After pair generation")

# Filter pairs for M-DRAC 
print("\nFiltering pairs for M-DRAC...")
mdrac_pairs = get_mdrac_pairs(base_pairs, config, skip_pair_generation=True)
print(f"✓ M-DRAC pairs after filtering: {len(mdrac_pairs):,}")

# Detect conflicts from filtered pairs
print("\nDetecting M-DRAC conflicts...")
mdrac_detector = ModifiedDRAC(config)
mdrac_conflicts = mdrac_detector.detect(mdrac_pairs, is_pairs_data=True)

print(f"\n{'='*70}")
print(f"M-DRAC Conflicts: {len(mdrac_conflicts):,}")
print(f"{'='*70}")


M-DRAC Detection

Generating nearby pairs...
[MEMORY] Before pair generation: 10626.6 MB
  Filtered vehicles: 1,227,630
  Generating pairs (max_distance=8.0m)...
  Processing 977,911 timestamps (batch_size=5,000)


  Pair generation: 100%|██████████| 196/196 [00:03<00:00, 49.22it/s]


  Applying overlap filter (buffer=0.5m)...
  ✓ Generated 138,800 nearby pairs (after overlap filter)
  ✓ Added zone information (zone1/zone2 columns)
✓ Generated 138,800 base pairs
[MEMORY] After pair generation: 10664.1 MB

Filtering pairs for M-DRAC...
 Approaching pairs: 53,032
  Zone filter (same lane): 47,370 pairs (filtered 5,662 different-lane)
  Lateral filter (<= 3.0m): 12,385 pairs (filtered 34,985 not aligned)
  ✓ Total filtered: 40,647 pairs | Remaining: 12,385 pairs
  Speed diff > 0.5: 3,908
  Final MDRAC pairs: 26
✓ M-DRAC pairs after filtering: 26

Detecting M-DRAC conflicts...
 Approaching pairs: 26
  Zone filter (same lane): 26 pairs (filtered 0 different-lane)
  Lateral filter (<= 3.0m): 26 pairs (filtered 0 not aligned)
  ✓ Total filtered: 0 pairs | Remaining: 26 pairs
  Speed diff > 0.5: 26
  Final MDRAC pairs: 26

M-DRAC Conflicts: 0


In [14]:
# Save M-DRAC results
if len(mdrac_conflicts) > 0:
    mdrac_path = save_detection_results(mdrac_conflicts, OUTPUT_DIR, 'mdrac', 'oulu', START_DATE)
    print(f"✓ Saved to {mdrac_path}")
    
    # Show sample
    print("\nSample conflicts:")
    print(mdrac_conflicts[['timestamp', 'label1', 'label2', 'mdrac', 'ttc', 'distance']].head(10))
else:
    print("No conflicts detected.")

No conflicts detected.


## Summary

In [15]:
print("\n" + "="*70)
print("OULU ANALYSIS COMPLETE")
print("="*70)
print(f"Date: {START_DATE} to {END_DATE}")
print(f"Final objects in near-miss zones: {len(df_analysis):,}")
print(f"M-DRAC conflicts detected: {len(mdrac_conflicts):,}")
print("="*70)


OULU ANALYSIS COMPLETE
Date: 2025-08-22 to 2025-09-11
Final objects in near-miss zones: 4,704,167
M-DRAC conflicts detected: 0
